# Fabric Classifier — run the web app on Colab

1. **Runtime → Run all** (or press ▶ on each cell in order). No GPU needed.
2. The last cell prints a link — click it to open the app in a new tab.
3. The link works while this notebook stays open. Re-run the last cell if the tab was closed.

In [ ]:
# 1. Get the latest code (public repo, no login needed) — always matches GitHub exactly
import os
os.chdir("/content")
if not os.path.exists("Fabric-Detection"):
    !git clone -q https://github.com/Adithya-109/Fabric-Detection.git
os.chdir("/content/Fabric-Detection")
!git fetch -q origin && git reset -q --hard origin/main
!git log --oneline -1

In [ ]:
# 2. Install the small packages Colab doesn't always have (PyTorch is preinstalled)
!pip install -q flask psutil transformers

In [ ]:
# 3. Start the web app in the background and wait until the model is loaded
#    (first start downloads CLIP, ~600 MB — allow a couple of minutes)
import subprocess, time, urllib.request
# stop any server left over from an earlier run (otherwise it keeps serving the OLD app on this port)
!pkill -f run_web.py || true
!fuser -k 5050/tcp || true
time.sleep(2)
log = open("/content/webapp.log", "w")
server = subprocess.Popen(["python", "run_web.py"], env={**os.environ, "PORT": "5050"},
                          stdout=log, stderr=subprocess.STDOUT)
for _ in range(480):
    try:
        urllib.request.urlopen("http://127.0.0.1:5050/api/health", timeout=2)
        print("✅ Server is up")
        break
    except Exception:
        if server.poll() is not None:
            break
        time.sleep(1)
else:
    print("Server did not start in time.")
if server.poll() is not None:
    print("❌ Server stopped. Last log lines:")
    print(open("/content/webapp.log").read()[-3000:])

In [ ]:
# 4. Open the app — click the link below
from google.colab import output
from google.colab.output import eval_js
print("App link:", eval_js("google.colab.kernel.proxyPort(5050)"))
output.serve_kernel_port_as_window(5050, anchor_text="👉 Open the Fabric Classifier app")

**Troubleshooting:** if the app shows an error, run `!tail -50 /content/webapp.log` in a new cell and share the output.